
# 🐱 Cats vs Dogs on OpenShift AI 🐶

## A short, fun computer-vision workshop

This notebook is designed to make you comfortable using an **OpenShift AI workbench**.

You will:

1. check the Python/GPU environment;
2. download a small cats-vs-dogs dataset from Kaggle;
3. explore a few images;
4. use **MobileNetV2 transfer learning**;
5. train a small classifier;
6. measure validation accuracy;
7. give the model a **mystery image**;
8. see whether it predicts **Cat** or **Dog** correctly;
9. save the trained model.

The dataset is intentionally small, so the notebook should run quickly.


## 1. Check the OpenShift AI workbench

In [ ]:

import os
import platform
import sys

print("Python:", sys.version.split()[0])
print("Hostname:", platform.node())
print("Working directory:", os.getcwd())
print("CPU cores:", os.cpu_count())


## 2. Check/install the small workshop dependencies

In [ ]:

import importlib.util
import subprocess
import sys

# Keep the environment changes minimal.
if importlib.util.find_spec("kagglehub") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "kagglehub"
    ])

# torchvision 0.21 matches PyTorch 2.6.
if importlib.util.find_spec("torchvision") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torchvision==0.21.0+cu124",
        "--index-url", "https://pypi.org/simple",
        "--extra-index-url", "https://download.pytorch.org/whl/cu124",
    ])

print("✓ Dependencies ready")


## 3. Verify PyTorch and the GPU

In [ ]:

import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

    free, total = torch.cuda.mem_get_info()

    print(f"GPU total: {total / 1024**3:.2f} GiB")
    print(f"GPU free:  {free / 1024**3:.2f} GiB")

    if free / 1024**3 < 1.0:
        print(
            "\n⚠ Less than 1 GiB is free. "
            "An old notebook kernel may still be using the GPU."
        )
        print(
            "This notebook will fall back to CPU unless GPU memory is freed."
        )
        device = torch.device("cpu")

print("Training device:", device)


## 4. Download the Kaggle dataset

In [ ]:

import kagglehub
from pathlib import Path

dataset_path = Path(
    kagglehub.dataset_download("marquis03/cats-and-dogs")
)

TRAIN_PATH = dataset_path / "train"
VAL_PATH = dataset_path / "val"

print("Dataset:", dataset_path)
print("Training folder:", TRAIN_PATH)
print("Validation folder:", VAL_PATH)

if not TRAIN_PATH.exists() or not VAL_PATH.exists():
    raise RuntimeError(
        "Expected train/ and val/ directories were not found.\n"
        f"Dataset contents: {[p.name for p in dataset_path.iterdir()]}"
    )


## 5. Load the images

In [ ]:

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMG_SIZE = 160
BATCH_SIZE = 32

# MobileNetV2 ImageNet normalization.
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ToTensor(),
    normalize,
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize,
])

train_dataset = datasets.ImageFolder(
    TRAIN_PATH,
    transform=train_transform,
)

val_dataset = datasets.ImageFolder(
    VAL_PATH,
    transform=val_transform,
)

class_names = train_dataset.classes

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

print("Classes:", class_names)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))


## 6. Explore a few images

In [ ]:

import matplotlib.pyplot as plt
from PIL import Image
import random

# Use the original image files for display.
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for row, class_name in enumerate(class_names):
    class_dir = TRAIN_PATH / class_name

    image_files = [
        p for p in class_dir.iterdir()
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]

    chosen = random.sample(
        image_files,
        min(4, len(image_files))
    )

    for col, image_path in enumerate(chosen):
        image = Image.open(image_path).convert("RGB")

        axes[row, col].imshow(image)
        axes[row, col].set_title(class_name)
        axes[row, col].axis("off")

plt.suptitle("A Few Training Images")
plt.tight_layout()
plt.show()



## 7. Load MobileNetV2

We use **transfer learning**:

- MobileNetV2 already knows useful visual features from ImageNet.
- We **freeze** the pretrained feature extractor.
- We train only a tiny final classifier for Cat vs Dog.

This makes training much faster than training a large CNN from scratch.


In [ ]:

import torch.nn as nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

weights = MobileNet_V2_Weights.DEFAULT

model = mobilenet_v2(weights=weights)

# Freeze the pretrained feature extractor.
for parameter in model.features.parameters():
    parameter.requires_grad = False

# Replace the ImageNet classifier with a 2-class head.
in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    in_features,
    len(class_names),
)

model = model.to(device)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print("Model device:", next(model.parameters()).device)


## 8. Train the classifier

In [ ]:

import time

EPOCHS = 5

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=1e-3,
)

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

start_time = time.perf_counter()

for epoch in range(1, EPOCHS + 1):

    # ---- training ----
    model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_count = 0

    for images, labels in train_loader:

        images = images.to(
            device,
            non_blocking=True,
        )

        labels = labels.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * len(images)

        train_correct += (
            logits.argmax(dim=1) == labels
        ).sum().item()

        train_count += len(images)

    train_loss = train_loss_sum / train_count
    train_acc = train_correct / train_count

    # ---- validation ----
    model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_count = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True,
            )

            labels = labels.to(
                device,
                non_blocking=True,
            )

            logits = model(images)
            loss = loss_fn(logits, labels)

            val_loss_sum += loss.item() * len(images)

            val_correct += (
                logits.argmax(dim=1) == labels
            ).sum().item()

            val_count += len(images)

    val_loss = val_loss_sum / val_count
    val_acc = val_correct / val_count

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train acc: {train_acc:.1%} | "
        f"val acc: {val_acc:.1%}"
    )

elapsed = time.perf_counter() - start_time

print(f"\n✓ Training finished in {elapsed:.1f} seconds")


## 9. Plot training progress

In [ ]:

epochs = range(1, EPOCHS + 1)

plt.figure(figsize=(9, 4))

plt.plot(
    epochs,
    history["train_acc"],
    marker="o",
    label="Training",
)

plt.plot(
    epochs,
    history["val_acc"],
    marker="o",
    label="Validation",
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Cat vs Dog Classification Accuracy")
plt.ylim(0, 1)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print(
    f"Final validation accuracy: "
    f"{history['val_acc'][-1]:.1%}"
)



# 🎯 Mystery Image Challenge

The next cell chooses **one random validation image**.

First look at the image and make your own guess.

Then run the prediction cell and see whether the model gets it right.


In [ ]:

import random

mystery_path, mystery_true_index = random.choice(
    val_dataset.samples
)

mystery_true_class = class_names[
    mystery_true_index
]

mystery_image = Image.open(
    mystery_path
).convert("RGB")

plt.figure(figsize=(6, 6))
plt.imshow(mystery_image)
plt.axis("off")
plt.title("Mystery Image — Cat or Dog?")
plt.show()

print("Make your guess before running the next cell!")


## 10. Ask the model

In [ ]:

import torch.nn.functional as F

model.eval()

input_tensor = val_transform(
    mystery_image
).unsqueeze(0).to(device)

with torch.no_grad():

    logits = model(input_tensor)

    probabilities = F.softmax(
        logits,
        dim=1,
    )[0]

    predicted_index = int(
        probabilities.argmax().item()
    )

    predicted_class = class_names[
        predicted_index
    ]

    confidence = float(
        probabilities[predicted_index].item()
    )

correct = (
    predicted_index == mystery_true_index
)

plt.figure(figsize=(6, 6))
plt.imshow(mystery_image)
plt.axis("off")

plt.title(
    f"Prediction: {predicted_class.upper()}\n"
    f"Confidence: {confidence:.1%}\n"
    f"Actual: {mystery_true_class.upper()}"
)

plt.show()

print("Predicted:", predicted_class)
print("Actual:   ", mystery_true_class)
print(f"Confidence: {confidence:.1%}")

if correct:
    print("\n✅ The model got it RIGHT!")
else:
    print("\n❌ The model got it WRONG!")



## 11. Optional: test your own image

Upload a `.jpg`, `.jpeg`, or `.png` file into the **same Jupyter folder** as this notebook.

Then set `MY_IMAGE` below, for example:

```python
MY_IMAGE = "my_cat.jpg"
```

and run the cell.


In [ ]:

# Change this filename after uploading your own image.
MY_IMAGE = None
# Example:
# MY_IMAGE = "my_cat.jpg"

if MY_IMAGE is None:

    print(
        "Upload an image into Jupyter, "
        "set MY_IMAGE to its filename, then run this cell again."
    )

else:

    image_path = Path(MY_IMAGE)

    if not image_path.exists():
        raise FileNotFoundError(image_path)

    user_image = Image.open(
        image_path
    ).convert("RGB")

    user_tensor = val_transform(
        user_image
    ).unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():

        logits = model(user_tensor)

        probabilities = F.softmax(
            logits,
            dim=1,
        )[0]

        predicted_index = int(
            probabilities.argmax().item()
        )

        predicted_class = class_names[
            predicted_index
        ]

        confidence = float(
            probabilities[predicted_index].item()
        )

    plt.figure(figsize=(6, 6))
    plt.imshow(user_image)
    plt.axis("off")
    plt.title(
        f"Prediction: {predicted_class.upper()}\n"
        f"Confidence: {confidence:.1%}"
    )
    plt.show()

    print("Prediction:", predicted_class)
    print(f"Confidence: {confidence:.1%}")


## 12. Save the trained model

In [ ]:

from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

model_file = output_dir / "cats_vs_dogs_mobilenetv2.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "class_names": class_names,
        "image_size": IMG_SIZE,
    },
    model_file,
)

print("Saved:", model_file)



# What did we just do?

```text
OpenShift AI Workbench
        ↓
Jupyter Notebook
        ↓
Kaggle dataset
        ↓
Explore images
        ↓
MobileNetV2 transfer learning
        ↓
Train Cat vs Dog classifier
        ↓
Validate accuracy
        ↓
Mystery image
        ↓
CAT or DOG?
        ↓
Save trained model
```

## Key OpenShift AI takeaway

You used your browser to run a complete computer-vision experiment in an OpenShift AI workbench:

- downloaded data;
- used Python libraries;
- used an accelerator when available;
- trained a model;
- visualised results;
- ran inference;
- saved a research artifact.

The same workflow can later grow into larger datasets, experiment tracking, pipelines and model serving.
